# PubMedQA LoRA+ on Modal

Primary fine-tuning notebook for running PubMedQA LoRA+ training inside a Modal notebook.

Assumptions:
- one strong GPU such as `A100`, `H100`, or `L40S`
- one persistent Modal Volume mounted at `PUBMEDQA_MODAL_ROOT`
- `HF_TOKEN` available in the Modal environment
- optional `WANDB_API_KEY`, `ADAPTER_HUB_REPO_ID`, and `MERGED_HUB_REPO_ID`


In [ ]:
import subprocess
import sys
from pathlib import Path

# Install the exact versions recorded in uv.lock. Pinning keeps the run
# reproducible: an unpinned `pip install -U` silently changes torch /
# transformers / peft under the notebook and invalidates the record.
PINNED = Path.cwd() / "requirements.lock.txt"
if not PINNED.exists():
    raise FileNotFoundError(
        f"{PINNED} is missing. Regenerate it with: uv export --frozen "
        "--no-hashes --no-emit-project --no-dev -o requirements.lock.txt"
    )
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PINNED)]
)


In [ ]:

import json
import os
import sys
import warnings
from pathlib import Path
from typing import Any

import torch
import wandb
from datasets import load_dataset
from huggingface_hub import login, snapshot_download

warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.pubmedqa_loraplus.config import LoRAPlusConfig

CONFIG_PATH = REPO_ROOT / "configs" / "loraplus_train.json"
DEFAULT_CONFIG = (
    json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    if CONFIG_PATH.exists()
    else {
        "run_name": "pubmedqa-loraplus",
        "base_model_id": "mistralai/Mistral-7B-Instruct-v0.3",
        "dataset_id": "qiaojin/PubMedQA",
        "dataset_config": "pqa_labeled",
        "model_dtype": "auto",
        "train_batch_size": 4,
        "eval_batch_size": 4,
        "gradient_accumulation_steps": 4,
        "learning_rate": 5e-5,
        "loraplus_lr_ratio": 16.0,
        "loraplus_lr_embedding": 1e-6,
        "num_epochs": 3,
        "logging_steps": 10,
        "save_steps": 200,
        "eval_steps": 200,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "target_modules": [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        "warmup_ratio": 0.03,
        "weight_decay": 0.0,
        "lr_scheduler_type": "cosine",
        "group_by_length": True,
        "max_grad_norm": 1.0,
        "max_length": 1024,
        "test_size": 0.1,
        "seed": 42,
        "mmlu_eval_samples": 50,
        "wandb_project": "pubmedqa-loraplus",
        "hub_token_env": "HF_TOKEN",
        "modal_mount_root": "/vol/pubmedqa-loraplus",
    }
)


def env_int(name: str, default: int) -> int:
    return int(os.getenv(name, str(default)))


def env_float(name: str, default: float) -> float:
    return float(os.getenv(name, str(default)))


if os.getenv("HF_TOKEN"):
    login(
        token=os.environ["HF_TOKEN"],
        add_to_git_credential=False,
    )

VOLUME_ROOT = Path(
    os.getenv(
        "PUBMEDQA_MODAL_ROOT",
        DEFAULT_CONFIG.get("modal_mount_root", "/vol/pubmedqa-loraplus"),
    )
)
RUN_NAME = os.getenv(
    "RUN_NAME",
    DEFAULT_CONFIG.get("run_name", "pubmedqa-loraplus"),
)
BASE_MODEL_DIR = VOLUME_ROOT / "base_model"
DATASET_DIR = VOLUME_ROOT / "dataset"
RUN_DIR = VOLUME_ROOT / "runs" / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
ADAPTER_DIR = RUN_DIR / "adapter"
TOKENIZER_DIR = RUN_DIR / "tokenizer"
MERGED_DIR = RUN_DIR / "merged"
REPORT_DIR = VOLUME_ROOT / "reports"
for path in [
    VOLUME_ROOT,
    BASE_MODEL_DIR,
    DATASET_DIR.parent,
    RUN_DIR,
    CHECKPOINT_DIR,
    REPORT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

CONFIG = {
    **DEFAULT_CONFIG,
    "run_name": RUN_NAME,
    "base_model_id": os.getenv(
        "BASE_MODEL_ID",
        DEFAULT_CONFIG["base_model_id"],
    ),
    "base_model_path": str(BASE_MODEL_DIR),
    "dataset_id": DEFAULT_CONFIG["dataset_id"],
    "dataset_config": DEFAULT_CONFIG["dataset_config"],
    "dataset_disk_path": str(DATASET_DIR),
    "output_root": str(RUN_DIR),
    "reports_root": str(REPORT_DIR),
    "modal_mount_root": str(VOLUME_ROOT),
    "model_dtype": os.getenv(
        "MODEL_DTYPE",
        DEFAULT_CONFIG.get("model_dtype", "auto"),
    ),
    "train_batch_size": env_int(
        "TRAIN_BATCH_SIZE",
        DEFAULT_CONFIG["train_batch_size"],
    ),
    "eval_batch_size": env_int(
        "EVAL_BATCH_SIZE",
        DEFAULT_CONFIG["eval_batch_size"],
    ),
    "gradient_accumulation_steps": env_int(
        "GRADIENT_ACCUMULATION_STEPS",
        DEFAULT_CONFIG["gradient_accumulation_steps"],
    ),
    "learning_rate": env_float(
        "LEARNING_RATE",
        DEFAULT_CONFIG["learning_rate"],
    ),
    "loraplus_lr_ratio": env_float(
        "LORAPLUS_LR_RATIO",
        DEFAULT_CONFIG["loraplus_lr_ratio"],
    ),
    "loraplus_lr_embedding": env_float(
        "LORAPLUS_LR_EMBEDDING",
        DEFAULT_CONFIG["loraplus_lr_embedding"],
    ),
    "num_epochs": env_int(
        "NUM_EPOCHS",
        DEFAULT_CONFIG["num_epochs"],
    ),
    "logging_steps": env_int(
        "LOGGING_STEPS",
        DEFAULT_CONFIG["logging_steps"],
    ),
    "save_steps": env_int(
        "SAVE_STEPS",
        DEFAULT_CONFIG["save_steps"] ,
    ),
    "eval_steps": env_int(
        "EVAL_STEPS",
        DEFAULT_CONFIG["eval_steps"],
    ),
    "max_length": env_int(
        "MAX_LENGTH",
        DEFAULT_CONFIG["max_length"],
    ),
    "mmlu_eval_samples": env_int(
        "MMLU_EVAL_SAMPLES",
        DEFAULT_CONFIG["mmlu_eval_samples"],
    ),
    "wandb_project": os.getenv(
        "WANDB_PROJECT",
        DEFAULT_CONFIG.get("wandb_project", "pubmedqa-loraplus"),
    ),
    "wandb_run_name": RUN_NAME,
    "adapter_hub_repo_id": os.getenv(
        "ADAPTER_HUB_REPO_ID",
        DEFAULT_CONFIG.get("adapter_hub_repo_id"),
    ),
    "merged_hub_repo_id": os.getenv(
        "MERGED_HUB_REPO_ID",
        DEFAULT_CONFIG.get("merged_hub_repo_id"),
    ),
}

os.environ.setdefault("WANDB_PROJECT", CONFIG["wandb_project"])
CONFIG_OBJ = LoRAPlusConfig(**CONFIG)
CONFIG_SNAPSHOT_PATH = REPORT_DIR / f"{RUN_NAME}_modal_notebook_config.json"
CONFIG_SNAPSHOT_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")

print(
    {
        "repo_root": str(REPO_ROOT),
        "modal_root": str(VOLUME_ROOT),
        "gpu_name": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else "cpu"
        ),
        "device_count": torch.cuda.device_count(),
        "base_model_id": CONFIG["base_model_id"],
        "run_name": RUN_NAME,
    }
)
if not torch.cuda.is_available():
    raise RuntimeError("This notebook is intended to run on a Modal GPU instance.")

In [ ]:

if DATASET_DIR.exists() and any(DATASET_DIR.iterdir()):
    print(f"Using cached dataset at {DATASET_DIR}")
else:
    dataset_source = load_dataset(CONFIG["dataset_id"], CONFIG["dataset_config"])
    dataset_source.save_to_disk(str(DATASET_DIR))
    print(f"Saved dataset to {DATASET_DIR}")


def base_model_is_complete(model_dir: Path) -> bool:
    index_path = model_dir / "model.safetensors.index.json"
    if index_path.exists():
        index_payload = json.loads(index_path.read_text(encoding="utf-8"))
        weight_map = index_payload.get("weight_map", {})
        required_files = sorted(set(weight_map.values()))
        missing_files = [name for name in required_files if not (model_dir / name).exists()]
        if missing_files:
            print("Base model cache is incomplete; missing shards:", missing_files)
            return False
        return True

    fallback_files = [
        model_dir / "consolidated.safetensors",
        model_dir / "model.safetensors",
        model_dir / "pytorch_model.bin",
    ]
    return any(path.exists() for path in fallback_files)


if base_model_is_complete(BASE_MODEL_DIR):
    print(f"Using cached base model at {BASE_MODEL_DIR}")
else:
    snapshot_download(
        repo_id=CONFIG["base_model_id"],
        local_dir=str(BASE_MODEL_DIR),
        token=(
            os.getenv(CONFIG.get("hub_token_env", "HF_TOKEN"))
            or None
        ),
    )
    print(f"Saved base model to {BASE_MODEL_DIR}")

from src.pubmedqa_loraplus.data import load_pubmedqa

# Same train/validation/test split the scripts use: checkpoint selection runs
# on `validation`, and the reported metrics come from the disjoint `test` set.
dataset = load_pubmedqa(CONFIG_OBJ)
print(dataset)
print(dataset["train"][0])


In [ ]:
# Training, evaluation and scoring live in src/pubmedqa_loraplus. This
# notebook used to carry a second copy of them, which drifted from the
# package: it returned 0.0 instead of None for unevaluated MMLU subjects and
# computed the eval warnings differently. Importing keeps one implementation.
from src.pubmedqa_loraplus.artifact import prepare_adapter_for_publish
from src.pubmedqa_loraplus.data import (
    CompletionOnlyCollator,
    tokenize_dataset,
)
from src.pubmedqa_loraplus.evaluation import run_evaluation_suite
from src.pubmedqa_loraplus.report import build_run_report
from src.pubmedqa_loraplus.workflow import (
    LoRAPlusTrainer,
    apply_lora_plus_adapter,
    build_training_arguments,
    build_training_summary,
    export_artifacts,
    load_base_model,
    load_tokenizer,
    set_seed,
)


def get_wandb_metadata() -> dict[str, Any]:
    run = getattr(wandb, "run", None)
    if run is None:
        return {}
    path = getattr(run, "path", None)
    if isinstance(path, (list, tuple)):
        path = "/".join(str(part) for part in path)
    return {
        "url": getattr(run, "url", None),
        "name": getattr(run, "name", None),
        "id": getattr(run, "id", None),
        "project": getattr(run, "project", None),
        "entity": getattr(run, "entity", None),
        "path": path,
    }


def flatten_numeric_metrics(prefix: str, payload: dict[str, Any]) -> dict[str, float]:
    metrics: dict[str, float] = {}
    for key, value in payload.items():
        metric_key = f"{prefix}/{key}" if prefix else str(key)
        if isinstance(value, dict):
            metrics.update(flatten_numeric_metrics(metric_key, value))
        elif isinstance(value, (int, float)) and not isinstance(value, bool):
            metrics[metric_key] = float(value)
    return metrics


print("package imports resolved")


In [ ]:
set_seed(CONFIG_OBJ.seed)
tokenizer = load_tokenizer(CONFIG_OBJ)
tokenized_dataset = tokenize_dataset(dataset, tokenizer, CONFIG_OBJ)

model = apply_lora_plus_adapter(load_base_model(CONFIG_OBJ), CONFIG_OBJ)
trainer = LoRAPlusTrainer(
    loraplus_config=CONFIG_OBJ,
    model=model,
    args=build_training_arguments(CONFIG_OBJ),
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=CompletionOnlyCollator(tokenizer.pad_token_id),
)

train_result = trainer.train(
    resume_from_checkpoint=CONFIG_OBJ.resume_from_checkpoint
)
trainer.save_state()
CONFIG_OBJ.adapter_output_dir.mkdir(parents=True, exist_ok=True)
CONFIG_OBJ.tokenizer_output_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(CONFIG_OBJ.adapter_output_dir))
tokenizer.save_pretrained(str(CONFIG_OBJ.tokenizer_output_dir))

# PEFT records the base model exactly as it was loaded. This notebook loads it
# from the Modal volume, so rewrite the reference to the public Hub id before
# the adapter is published, and drop the Trainer bookkeeping pickle.
prepare_adapter_for_publish(CONFIG_OBJ.adapter_output_dir, CONFIG["base_model_id"])

# One report schema shared with the scripts and the model card. The card reads
# reports/<RUN_NAME>.json, so the training metadata goes there and the
# evaluation cell merges its metrics into the same payload.
training_payload = build_run_report(
    run_name=RUN_NAME,
    base_model_id=CONFIG["base_model_id"],
    base_model_source=str(BASE_MODEL_DIR),
    dataset_id=CONFIG["dataset_id"],
    dataset_config=CONFIG["dataset_config"],
    dataset_disk_path=str(DATASET_DIR),
    train_examples=len(dataset["train"]),
    validation_examples=len(dataset["validation"]),
    eval_examples=len(dataset["test"]),
    training={
        "learning_rate": CONFIG["learning_rate"],
        "loraplus_lr_ratio": CONFIG["loraplus_lr_ratio"],
        "num_epochs": CONFIG["num_epochs"],
        "train_batch_size": CONFIG["train_batch_size"],
        "gradient_accumulation_steps": CONFIG["gradient_accumulation_steps"],
        "best_model_checkpoint": trainer.state.best_model_checkpoint,
        "global_step": trainer.state.global_step,
        "train_metrics": dict(train_result.metrics),
        "wandb": get_wandb_metadata(),
    },
    training_summary=build_training_summary(trainer, train_result),
    artifacts={
        "checkpoints": str(CONFIG_OBJ.checkpoints_dir),
        "adapter": str(CONFIG_OBJ.adapter_output_dir),
        "tokenizer": str(CONFIG_OBJ.tokenizer_output_dir),
    },
    evaluation={},
)
CONFIG_OBJ.report_path.parent.mkdir(parents=True, exist_ok=True)
CONFIG_OBJ.report_path.write_text(
    json.dumps(training_payload, indent=2),
    encoding="utf-8",
)
CONFIG_OBJ.training_log_path.write_text(
    json.dumps(trainer.state.log_history, indent=2),
    encoding="utf-8",
)
training_payload


In [ ]:
from src.pubmedqa_loraplus.workflow import load_model_for_artifact

adapter_model, _ = load_model_for_artifact(CONFIG_OBJ, "adapter")
evaluation = run_evaluation_suite(
    adapter_model,
    tokenizer,
    dataset["test"],
    CONFIG_OBJ,
)

if getattr(wandb, "run", None) is not None:
    wandb.log(flatten_numeric_metrics("evaluation", evaluation))

# Merge the metrics into the training payload rather than writing a second,
# differently shaped report: build_model_card reads this one file.
evaluation_payload = json.loads(CONFIG_OBJ.report_path.read_text(encoding="utf-8"))
evaluation_payload["evaluation"] = evaluation
evaluation_payload["wandb"] = get_wandb_metadata()
CONFIG_OBJ.report_path.write_text(
    json.dumps(evaluation_payload, indent=2),
    encoding="utf-8",
)
evaluation_payload


In [ ]:
export_payload = export_artifacts(CONFIG_OBJ)
export_payload["hub"] = {
    "adapter_hub_repo_id": CONFIG.get("adapter_hub_repo_id"),
    "merged_hub_repo_id": CONFIG.get("merged_hub_repo_id"),
}
(CONFIG_OBJ.reports_root / f"{RUN_NAME}_exports.json").write_text(
    json.dumps(export_payload, indent=2),
    encoding="utf-8",
)
export_payload


In [ ]:
# Optional: publish the artifacts. Skipped unless a repo id is configured.
# Uses the same code path as scripts/push_to_hub.py. The Hub repo is created
# private unless `hub_private` is explicitly False in the config.
from src.pubmedqa_loraplus.hub import push_artifact_to_hub

for artifact in ("adapter", "merged"):
    if artifact == "adapter":
        repo_id = CONFIG_OBJ.adapter_hub_repo_id
    else:
        repo_id = CONFIG_OBJ.merged_hub_repo_id
    if not repo_id:
        print(f"Skipping {artifact}: no Hub repo id configured")
        continue
    print(push_artifact_to_hub(CONFIG_OBJ, artifact))
